In [1]:
from MIDI_AUTOMATION.constants import voice_struct, VOICE_KEYS, checksum

def reorder_operator_block(block):
    # block: length 21
    block = block[:] + [0]
    new_order = [
        0, 1, 2, 3,    # R1-R4
        4, 5, 6, 7,    # L1-L4
        8, 9, 10,      # BP, LD, RD
        12, 11,        # RC, LC
        20, 13,        # DET, RS
        15, 14,        # KVS, AMS
        16,            # OL
        18, 17,        # FC, M
        19             # FF
    ]

    block = block[new_order]
    block[13] = 0
    return block

def transform_params(arr):
    arr = np.asarray(arr)
    operators = arr[:126].reshape(6, 21)  # 6*21=126
    tail = arr[126:]  # remaining global params

    new_order = [
        0, 1, 2, 3,    # PG1-PG4
        4, 5, 6, 7,    # PL1-PL4
        8, 10, 9,      # ALG, OSC, FB
        11, 12,        # LFS, LFD
        13, 14,        # LPMD, LAMD
        17, 16, 15,    # LPMS, LFW, LKS
        16            # TRNSP
    ]

    tail = tail[new_order]
    tail = np.append(tail, [0] * 10)

    reordered_ops = np.array([reorder_operator_block(op) for op in operators])
    return np.concatenate([reordered_ops.flatten(), tail])


def dx7_pack(voices):
    HEADER = int('0x43', 0), int('0x00', 0), int('0x09', 0), int('0x20', 0), int('0x00', 0)

    voices_bytes = bytes()
    for voice in voices:
        voice = transform_params(voice)
        voice_bytes = voice_struct.pack(dict(zip(VOICE_KEYS, voice)))
        voices_bytes += voice_bytes

    patch_checksum = [checksum(voices_bytes)]

    data = bytes(HEADER) + voices_bytes + bytes(patch_checksum)

    return mido.Message('sysex', data=data)

def undo_one_hot(processor, df_encoded):
    df_out = {}
    idx = 0

    for p in processor.params:

        if p.p_type == 'm':
            df_out[p.name] = df_encoded.iloc[:, idx]
            idx += 1

        elif p.p_type == 'b':
            cols = df_encoded.iloc[:, idx:idx+2].to_numpy()
            df_out[p.name] = cols.argmax(axis=1)
            idx += 2

        elif p.p_type in ('c', 'x'):
            cols = df_encoded.iloc[:, idx:idx+p.n_classes]
            df_out[p.name] = cols.values.argmax(axis=1)
            idx += p.n_classes

    return pd.DataFrame(df_out)

def denormalize(processor, df):
    df = df.copy()

    for p in processor.params:
        if p.p_type == 'm':
            df[p.name] = df[p.name] * p.p_max

    return df

def cast_patch_to_int(processor, df):
    df = df.copy()

    for p in processor.params:
        if p.p_type == 'm':
            df[p.name] = df[p.name].round().astype(int)
        else:
            df[p.name] = df[p.name].astype(int)

    df[df < 0] = 0

    return df

In [ ]:
import pretty_midi
import random

def generate_random_midi(path):
    midi = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=0)

    t = 0
    for _ in range(random.randint(6,12)):
        pitch = random.randint(36,84)
        dur = random.uniform(0.2,1.2)

        note = pretty_midi.Note(
            velocity=random.randint(60,120),
            pitch=pitch,
            start=t,
            end=t+dur
        )

        inst.notes.append(note)
        t += dur

    midi.instruments.append(inst)
    midi.write(path)

In [ ]:
mins = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

maxs = [99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 3, 3, 7, 3, 7, 99, 1, 31, 99, 14, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 3, 3, 7, 3, 7, 99, 1, 31, 99, 14, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 3, 3, 7, 3, 7, 99, 1, 31, 99, 14, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 3, 3, 7, 3, 7, 99, 1, 31, 99, 14, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 3, 3, 7, 3, 7, 99, 1, 31, 99, 14, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 3, 3, 7, 3, 7, 99, 1, 31, 99, 14, 99, 99, 99, 99, 99, 99, 99, 99, 31, 7, 1, 99, 99, 99, 99, 1, 5, 7, 48]

In [5]:
num_midi_files = 2 ** 15
for index in range(num_midi_files):
    generate_random_midi(f"./midi/{index:06d}.midi")